# Reddit annotated graph: preprocessing, NetworkX, and baselines

This notebook loads the annotated CSVs under `annotated/`, builds a **directed user–reply** NetworkX graph (edge `u → v` means user `u` replied to user `v`), constructs the **symmetrized normalized adjacency** used by the simple GCN baseline, and runs the same baselines as `run_reddit_baselines.py` with **repeated stratified 60% / 20% / 20%** train / validation / test splits (`test_size=0.4` then `0.5` on the remainder, `stratify=y`, `random_state=seed`).

**Inputs** (default: project `annotated/` folder):
- `nodes_annotated.csv`
- `comments_with_stances.csv`
- `annotation_template_annotated.csv`

In [9]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from IPython.display import display

# Repository root (parent of this notebook if notebook lives in project root)
PROJECT_ROOT = Path.cwd()
INPUT_DIR = PROJECT_ROOT / "annotated"

N_SEEDS = 10  # match run_reddit_baselines.py default

print("INPUT_DIR:", INPUT_DIR.resolve())

INPUT_DIR: /Users/shangao/Documents/GitHub/networkscienceproject/annotated


## 1. Load CSVs and preprocess

- Merge stance from `annotation_template_annotated.csv` when the node table already has a `stance` column (annotation wins when present).
- Drop rows missing `user_id` or `stance`.
- Normalize `author` / `parent_author` in comments for graph edges.

In [10]:
def load_data(input_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    nodes_path = input_dir / "nodes_annotated.csv"
    comments_path = input_dir / "comments_with_stances.csv"
    ann_path = input_dir / "annotation_template_annotated.csv"

    for p in (nodes_path, comments_path, ann_path):
        if not p.exists():
            raise FileNotFoundError(p)

    nodes = pd.read_csv(nodes_path)
    comments = pd.read_csv(comments_path)
    ann = pd.read_csv(ann_path)

    if "stance" not in nodes.columns:
        nodes = nodes.merge(ann[["user_id", "stance"]], on="user_id", how="left")
    else:
        ann_small = ann[["user_id", "stance"]].rename(columns={"stance": "stance_ann"})
        nodes = nodes.merge(ann_small, on="user_id", how="left")
        nodes["stance"] = nodes["stance_ann"].fillna(nodes["stance"])
        nodes = nodes.drop(columns=["stance_ann"])

    nodes = nodes.dropna(subset=["user_id", "stance"]).copy()
    nodes["user_id"] = nodes["user_id"].astype(str)
    nodes["all_text"] = nodes["all_text"].fillna("").astype(str)
    nodes["stance"] = nodes["stance"].astype(str)

    comments = comments.copy()
    comments["author"] = comments["author"].astype(str)
    comments["parent_author"] = comments["parent_author"].astype(str)

    return nodes, comments


nodes, comments = load_data(INPUT_DIR)
display(nodes.head(2))
print("Users (labeled nodes):", len(nodes))
print(nodes["stance"].value_counts())

,user_id,num_comments,all_text,stance,notes
0,Kybrator,27,&gt; life(unborn child) is more important than...,pro_life,Prototype label from aggregated comments; emph...
1,Genoscythe_,16,&amp;#x200B;\n\n&gt;People should have body au...,pro_choice,Prototype label from aggregated comments; argu...


Users (labeled nodes): 47
stance
pro_choice       25
pro_life         11
neutral_mixed    11
Name: count, dtype: int64


## 2. Build NetworkX `DiGraph` and GCN adjacency

- **Nodes**: each row in `nodes` (users with a stance label).
- **Edges**: aggregated reply counts from `comments` where both endpoints appear in the node table (`author` → `parent_author`).
- **GCN matrix**: symmetrize edge weights, add self-loops, then \(D^{-1/2} A D^{-1/2}\) (same as `run_reddit_baselines.py`).

In [11]:
def build_networkx_graph(
    nodes: pd.DataFrame, comments: pd.DataFrame
) -> tuple[nx.DiGraph, list[str], dict[str, int], np.ndarray]:
    """Return directed graph, node order, index map, and normalized adjacency for GCN."""
    user_ids = nodes["user_id"].tolist()
    user_to_idx = {u: i for i, u in enumerate(user_ids)}
    n = len(user_ids)

    G = nx.DiGraph()
    for _, row in nodes.iterrows():
        G.add_node(row["user_id"], stance=row["stance"], text=row["all_text"])

    edge_df = comments[["author", "parent_author"]].dropna()
    edge_df = edge_df[
        (edge_df["author"] != "")
        & (edge_df["parent_author"] != "")
        & (edge_df["author"] != "None")
        & (edge_df["parent_author"] != "None")
    ]

    grouped = edge_df.groupby(["author", "parent_author"]).size().reset_index(name="weight")
    for _, row in grouped.iterrows():
        src, dst = row["author"], row["parent_author"]
        if src in user_to_idx and dst in user_to_idx:
            G.add_edge(src, dst, weight=int(row["weight"]))

    A = np.zeros((n, n), dtype=np.float32)
    for u, v, data in G.edges(data=True):
        i, j = user_to_idx[u], user_to_idx[v]
        w = float(data.get("weight", 1.0))
        A[i, j] += w
        A[j, i] += w

    A += np.eye(n, dtype=np.float32)
    deg = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.clip(deg, 1e-8, None))).astype(np.float32)
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt

    return G, user_ids, user_to_idx, A_norm


G, user_ids, user_to_idx, A_norm = build_networkx_graph(nodes, comments)

print(G)
print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())
print("A_norm shape:", A_norm.shape)

# Example: neighbors of the first user in node table
u0 = user_ids[0]
print(f"Sample node {u0!r} out-degree:", G.out_degree(u0), "in-degree:", G.in_degree(u0))

DiGraph with 47 nodes and 78 edges
Nodes: 47 Edges: 78
A_norm shape: (47, 47)
Sample node 'Kybrator' out-degree: 15 in-degree: 11


## 3. Labels, TF–IDF + SVD features, and stratified splits

Split procedure (identical to `make_splits` in `run_reddit_baselines.py`):
1. `train_test_split(..., test_size=0.4, stratify=y, random_state=seed)` → train vs temp
2. `train_test_split(temp, test_size=0.5, stratify=y_temp, random_state=seed)` → val vs test

So **train ≈ 60%**, **val ≈ 20%**, **test ≈ 20%** of labeled users.

In [12]:
from transformers import AutoModel, AutoTokenizer


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom


def build_tinybert_embeddings(
    texts: list[str],
    model_name: str = "huawei-noah/TinyBERT_General_4L_312D",
    batch_size: int = 16,
    max_length: int = 256,
    device: str | None = None,
) -> np.ndarray:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name)
    mdl.eval()
    mdl.to(device)

    outs: list[np.ndarray] = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tok(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            out = mdl(**enc)
            emb = mean_pool(out.last_hidden_state, enc["attention_mask"])
            outs.append(emb.cpu().numpy().astype(np.float32))

    return np.vstack(outs)


def build_text_features(nodes: pd.DataFrame) -> np.ndarray:
    tinybert_cache = INPUT_DIR / "tinybert_embeddings.npy"
    texts = nodes["all_text"].fillna("").astype(str).tolist()
    if tinybert_cache.exists():
        X = np.load(tinybert_cache)
        if X.shape[0] == len(nodes):
            return X.astype(np.float32)
    X = build_tinybert_embeddings(texts)
    np.save(tinybert_cache, X)
    return X


def make_splits(y: np.ndarray, seed: int):
    idx = np.arange(len(y))
    train_idx, temp_idx, y_train, y_temp = train_test_split(
        idx, y, test_size=0.4, random_state=seed, stratify=y
    )
    val_idx, test_idx, _, _ = train_test_split(
        temp_idx, y_temp, test_size=0.5, random_state=seed, stratify=y_temp
    )
    return train_idx, val_idx, test_idx


label_names = sorted(nodes["stance"].unique())
label_to_id = {label: i for i, label in enumerate(label_names)}
id_to_label = {i: label for label, i in label_to_id.items()}

y = np.array([label_to_id[s] for s in nodes["stance"]], dtype=np.int64)
X_tinybert = build_text_features(nodes)
X = X_tinybert

print("X_tinybert shape:", X_tinybert.shape, "n_classes:", len(label_names))

train_idx, val_idx, test_idx = make_splits(y, seed=0)
print(
    "Seed 0 split sizes — train:", len(train_idx),
    "val:", len(val_idx),
    "test:", len(test_idx),
)

X shape: (47, 46) n_classes: 3
Seed 0 split sizes — train: 28 val: 9 test: 10


## 4. Baseline models (same as `run_reddit_baselines.py`)

- **majority**: majority class on train, predict for test
- **text_logreg_tinybert**: logistic regression on TinyBERT features
- **text_mlp_tinybert**: small MLP on TinyBERT features
- **gcn_tinybert**: 2-layer GCN with TinyBERT node features and early stopping on validation macro-F1

In [13]:
def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def metrics_dict(y_true, y_pred):
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }


def run_majority(y, train_idx, test_idx):
    majority = Counter(y[train_idx]).most_common(1)[0][0]
    pred = np.full(len(test_idx), majority)
    return pred, metrics_dict(y[test_idx], pred)


def run_text_logreg(X, y, train_idx, test_idx, seed):
    clf = LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=seed
    )
    clf.fit(X[train_idx], y[train_idx])
    pred = clf.predict(X[test_idx])
    return pred, metrics_dict(y[test_idx], pred)


def run_text_mlp(X, y, train_idx, test_idx, seed):
    clf = MLPClassifier(
        hidden_layer_sizes=(32,),
        activation="relu",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=1000,
        random_state=seed,
    )
    clf.fit(X[train_idx], y[train_idx])
    pred = clf.predict(X[test_idx])
    return pred, metrics_dict(y[test_idx], pred)


class GCN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout: float = 0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, adj):
        x = adj @ x
        x = self.fc1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = adj @ x
        x = self.fc2(x)
        return x


def run_gcn(X, A_norm, y, train_idx, val_idx, test_idx, seed):
    set_seed(seed)
    x = torch.tensor(X, dtype=torch.float32)
    adj = torch.tensor(A_norm, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)

    train_mask = torch.zeros(len(y), dtype=torch.bool)
    val_mask = torch.zeros(len(y), dtype=torch.bool)
    test_mask = torch.zeros(len(y), dtype=torch.bool)
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    model = GCN(
        in_dim=X.shape[1],
        hidden_dim=32,
        out_dim=len(np.unique(y)),
        dropout=0.4,
    )
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    best_state = None
    best_val_f1 = -1.0
    patience, wait = 50, 0

    for _ in range(300):
        model.train()
        opt.zero_grad()
        logits = model(x, adj)
        loss = F.cross_entropy(logits[train_mask], yt[train_mask])
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x, adj)[val_mask]
            val_pred = val_logits.argmax(dim=1).cpu().numpy()
            val_f1 = f1_score(y[val_idx], val_pred, average="macro")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        test_pred = model(x, adj)[test_mask].argmax(dim=1).cpu().numpy()

    return test_pred, metrics_dict(y[test_idx], test_pred)

In [14]:
all_rows = []
last_split_preds = []

for seed in range(N_SEEDS):
    train_idx, val_idx, test_idx = make_splits(y, seed)
    models = {
        "majority": lambda: run_majority(y, train_idx, test_idx),
        "text_logreg_tinybert": lambda: run_text_logreg(X, y, train_idx, test_idx, seed),
        "text_mlp_tinybert": lambda: run_text_mlp(X, y, train_idx, test_idx, seed),
        "gcn_tinybert": lambda: run_gcn(X, A_norm, y, train_idx, val_idx, test_idx, seed),
    }
    for model_name, runner in models.items():
        pred, mets = runner()
        all_rows.append(
            {
                "seed": seed,
                "model": model_name,
                "macro_f1": mets["macro_f1"],
                "accuracy": mets["accuracy"],
                "n_train": len(train_idx),
                "n_val": len(val_idx),
                "n_test": len(test_idx),
            }
        )
        if seed == N_SEEDS - 1:
            for idx, p in zip(test_idx, pred):
                last_split_preds.append(
                    {
                        "seed": seed,
                        "model": model_name,
                        "user_id": user_ids[idx],
                        "true_label": id_to_label[y[idx]],
                        "pred_label": id_to_label[int(p)],
                        "split": "test",
                    }
                )

results_df = pd.DataFrame(all_rows)
summary_df = (
    results_df.groupby("model")[["macro_f1", "accuracy"]]
    .agg(["mean", "std"])
    .round(4)
)
preds_df = pd.DataFrame(last_split_preds)

display(results_df.round(4))
display(summary_df)

,seed,model,macro_f1,accuracy,n_train,n_val,n_test
0,0,majority,0.2222,0.5,28,9,10
1,0,text_logreg,0.1905,0.4,28,9,10
2,0,text_mlp,0.0606,0.1,28,9,10
3,0,gcn,0.2222,0.5,28,9,10
4,1,majority,0.2222,0.5,28,9,10
5,1,text_logreg,0.1905,0.4,28,9,10
6,1,text_mlp,0.3000,0.4,28,9,10
7,1,gcn,0.2222,0.5,28,9,10
8,2,majority,0.2222,0.5,28,9,10
9,2,text_logreg,0.1905,0.4,28,9,10


macro_f1         accuracy        
                mean     std     mean     std
model                                        
gcn           0.2291  0.1316     0.42  0.1229
majority      0.2222  0.0000     0.50  0.0000
text_logreg   0.2113  0.0646     0.41  0.0568
text_mlp      0.2306  0.1064     0.42  0.1398

## 5. Optional: save outputs (same filenames as the CLI script)

Uncomment to write CSV/JSON next to the annotated data.

In [15]:
SAVE = False  # set True to write files

if SAVE:
    prefix = "baseline_results_notebook"
    results_df.to_csv(INPUT_DIR / f"{prefix}.csv", index=False)
    summary_df.to_csv(INPUT_DIR / f"{prefix}_summary.csv")
    preds_df.to_csv(INPUT_DIR / f"{prefix}_last_seed_test_predictions.csv", index=False)
    meta = {
        "label_to_id": label_to_id,
        "n_users": int(len(nodes)),
        "class_counts": nodes["stance"].value_counts().to_dict(),
        "split_scheme": "Repeated stratified 60/20/20",
        "n_seeds": N_SEEDS,
    }
    with open(INPUT_DIR / f"{prefix}_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    print("Wrote", prefix, "* under", INPUT_DIR)

## 6. TinyBERT text embeddings (node features)

This section computes **TinyBERT** embeddings from each user's `all_text` and caches them under `annotated/` so you only pay the cost once.

- Model: `huawei-noah/TinyBERT_General_4L_312D`
- Pooling: attention-mask mean pooling over token embeddings
- Output: `tinybert_embeddings.npy` aligned with `nodes` row order / `user_ids`

In [16]:
# If this import fails, install transformers in your environment/kernel:
#   pip install -U transformers sentencepiece
from transformers import AutoModel, AutoTokenizer


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom


def build_tinybert_embeddings(
    texts: list[str],
    model_name: str = "huawei-noah/TinyBERT_General_4L_312D",
    batch_size: int = 16,
    max_length: int = 256,
    device: str | None = None,
) -> np.ndarray:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name)
    mdl.eval()
    mdl.to(device)

    outs: list[np.ndarray] = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tok(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            out = mdl(**enc)
            emb = mean_pool(out.last_hidden_state, enc["attention_mask"])
            outs.append(emb.cpu().numpy().astype(np.float32))

    return np.vstack(outs)


tinybert_cache = INPUT_DIR / "tinybert_embeddings.npy"
texts = nodes["all_text"].fillna("").astype(str).tolist()

if tinybert_cache.exists():
    X_tinybert = np.load(tinybert_cache)
    print("Loaded cached TinyBERT embeddings:", tinybert_cache)
else:
    X_tinybert = build_tinybert_embeddings(texts)
    np.save(tinybert_cache, X_tinybert)
    print("Saved TinyBERT embeddings:", tinybert_cache)

print("X_tinybert shape:", X_tinybert.shape)

/Users/shangao/Documents/GitHub/networkscienceproject/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 71/71 [00:00<00:00, 18621.53it/s]
BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.prediction

Saved TinyBERT embeddings: /Users/shangao/Documents/GitHub/networkscienceproject/annotated/tinybert_embeddings.npy
X_tinybert shape: (47, 312)


## 7. GraphSAGE (heterophily-tolerant baseline)

GraphSAGE often works better than vanilla GCN when homophily assumptions break down.

**Features used**: TinyBERT embeddings (`X_tinybert`).

**Graph used**: undirected message passing on the reply graph (we add both directions for each edge).

In [17]:
# If this import fails, install PyTorch Geometric in your environment/kernel.
# Install varies by OS + torch version; start with:
#   pip install -U torch-geometric
# If that fails, follow the official wheel instructions for your torch version:
#   https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv


def build_pyg_data(G: nx.DiGraph, X_feat: np.ndarray, y: np.ndarray) -> Data:
    edges: list[tuple[int, int]] = []
    for u, v in G.edges():
        edges.append((user_to_idx[u], user_to_idx[v]))
        edges.append((user_to_idx[v], user_to_idx[u]))
    if len(edges) == 0:
        raise ValueError("Graph has 0 edges after filtering to labeled users.")
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(
        x=torch.tensor(X_feat, dtype=torch.float32),
        edge_index=edge_index,
        y=torch.tensor(y, dtype=torch.long),
    )


class GraphSAGE(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        dropout: float = 0.5,
        aggr: str = "max",
    ):
        super().__init__()
        # Using max aggregation is often a bit more robust under heterophily than mean.
        self.conv1 = SAGEConv(in_dim, hidden_dim, aggr=aggr)
        self.conv2 = SAGEConv(hidden_dim, out_dim, aggr=aggr)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


def run_graphsage(
    data: Data,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
    seed: int,
    hidden_dim: int = 64,
    lr: float = 0.01,
    weight_decay: float = 5e-4,
    dropout: float = 0.4,
    max_epochs: int = 500,
    patience: int = 50,
) -> tuple[np.ndarray, dict]:
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data = data.to(device)

    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool, device=device)
    val_mask = torch.zeros(data.num_nodes, dtype=torch.bool, device=device)
    test_mask = torch.zeros(data.num_nodes, dtype=torch.bool, device=device)
    train_mask[torch.tensor(train_idx, device=device)] = True
    val_mask[torch.tensor(val_idx, device=device)] = True
    test_mask[torch.tensor(test_idx, device=device)] = True

    model = GraphSAGE(
        in_dim=data.x.size(-1),
        hidden_dim=hidden_dim,
        out_dim=int(data.y.max().item() + 1),
        dropout=dropout,
    ).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val_f1 = -1.0
    wait = 0

    for _ in range(max_epochs):
        model.train()
        opt.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[train_mask], data.y[train_mask])
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = logits[val_mask].argmax(dim=1).cpu().numpy()
            val_f1 = f1_score(y[val_idx], val_pred, average="macro")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        test_logits = model(data.x, data.edge_index)[test_mask]
        test_pred = test_logits.argmax(dim=1).cpu().numpy()

    return test_pred, metrics_dict(y[test_idx], test_pred)


data_tinybert = build_pyg_data(G, X_tinybert, y)
print(data_tinybert)

sage_rows = []
for seed in range(N_SEEDS):
    train_idx, val_idx, test_idx = make_splits(y, seed)
    _, mets = run_graphsage(data_tinybert, train_idx, val_idx, test_idx, seed=seed)
    sage_rows.append({"seed": seed, "model": "graphsage_tinybert", **mets})

sage_df = pd.DataFrame(sage_rows)
display(sage_df.round(4))
display(sage_df[["macro_f1", "accuracy"]].agg(["mean", "std"]).round(4))

Data(x=[47, 312], edge_index=[2, 156], y=[47])


,seed,model,macro_f1,accuracy
0,0,graphsage_tinybert,0.3758,0.5
1,1,graphsage_tinybert,0.1905,0.4
2,2,graphsage_tinybert,0.4167,0.4
3,3,graphsage_tinybert,0.1538,0.3
4,4,graphsage_tinybert,0.1212,0.2
5,5,graphsage_tinybert,0.3333,0.4
6,6,graphsage_tinybert,0.2333,0.2
7,7,graphsage_tinybert,0.2381,0.5
8,8,graphsage_tinybert,0.3152,0.4
9,9,graphsage_tinybert,0.1111,0.2


,macro_f1,accuracy
mean,0.2489,0.3500
std,0.1074,0.1179
